In [10]:
import random
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms
from transformers import AutoFeatureExtractor, ViTForImageClassification
import optuna
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report

# Seed for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cpu


In [11]:
train_path = 'C:/Users/JARE WORKS/Documents/aj project/ovie results/archive (3)/Training'
test_path = 'C:/Users/JARE WORKS/Documents/aj project/ovie results/archive (3)/Testing'

valid_classes = {'notumor', 'meningioma', 'glioma', 'pituitary'}

extractor = AutoFeatureExtractor.from_pretrained('google/vit-base-patch32-224-in21k')

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=extractor.image_mean, std=extractor.image_std)
])

def filter_dataset(path, transform):
    base_dataset = datasets.ImageFolder(path, transform=transform)
    valid_indices = [i for i, (_, label) in enumerate(base_dataset.samples) if base_dataset.classes[label] in valid_classes]
    filtered_dataset = Subset(base_dataset, valid_indices)
    return filtered_dataset

train_ds_full = filter_dataset(train_path, transform)
test_ds = filter_dataset(test_path, transform)

val_size = int(0.2 * len(train_ds_full))
train_size = len(train_ds_full) - val_size
train_ds, val_ds = random_split(train_ds_full, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16)
test_loader = DataLoader(test_ds, batch_size=16)

class_names = ['glioma', 'meningioma', 'notumor', 'pituitary']


In [12]:
from transformers import ViTForImageClassification

def get_vit_b32(num_labels=4):
    model = ViTForImageClassification.from_pretrained('google/vit-base-patch32-224-in21k', num_labels=num_labels)
    return model


In [13]:
# Placeholder functions — you need to replace these with actual implementations or imports

def get_rvit(num_labels=4):
    # Implement or load RViT model here
    model = ...  # Your RViT model instance
    return model

def get_ftvt_i16(num_labels=4):
    # Implement or load FTVT-I16 model here
    model = ...  # Your FTVT-I16 model instance
    return model

def get_gvit_regularized_model(base_model):
    # Wrap or modify base_model with GViT regularization
    model = ...  # Your GViT-enhanced model
    return model

def apply_feature_selection_sca(model):
    # Apply Selective Cross-Attention
    return model

def apply_feature_calibration_fcm(model):
    # Apply Feature Calibration Module
    return model


In [14]:
import optuna

def objective(trial):
    lr = trial.suggest_float('lr', 1e-6, 1e-4, log=True)
    wd = trial.suggest_float('weight_decay', 0.0, 0.1)
    
    model = get_vit_b32()
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    criterion = nn.CrossEntropyLoss()
    
    scaler = torch.cuda.amp.GradScaler()
    
    # Simple training loop - just 1 epoch to speed up trial
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            outputs = model(images).logits
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        break  # Run only one batch to speed up
    
    # Evaluate on validation set
    model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images).logits
            preds = torch.argmax(outputs, dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())
    
    f1 = f1_score(val_labels, val_preds, average='macro')
    return f1

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

print("Best trial:")
print(study.best_trial.params)


[I 2025-06-03 09:52:16,256] A new study created in memory with name: no-name-fee66cd6-83a0-44de-ac47-40954f580f39
Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch32-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\JARE WORKS\AppData\Local\Temp\ipykernel_17664\2223371079.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
C:\Users\JARE WORKS\AppData\Local\Temp\ipykernel_17664\2223371079.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
[I 2025-06-03 09:53:35,535] Trial 0 finished with value: 0.21863339190660502 and parameters: {'lr': 4.0327417996

Best trial:
{'lr': 2.8632502135973155e-05, 'weight_decay': 0.05247180687001397}


In [15]:
def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            outputs = model(images).logits
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate_model(model, loader):
    model.eval()
    all_preds, all_labels, all_logits = [], [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images).logits
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_logits.extend(outputs.cpu().numpy())
    return np.array(all_preds), np.array(all_labels), np.array(all_logits)


In [16]:
def run_training_and_evaluation(model_name, get_model_func, apply_reg=False, apply_fs=None, epochs=5):
    print(f"Training {model_name}...")

    model = get_model_func()
    if apply_reg:
        model = get_gvit_regularized_model(model)
    if apply_fs == 'sca':
        model = apply_feature_selection_sca(model)
    elif apply_fs == 'fcm':
        model = apply_feature_calibration_fcm(model)
    model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
    criterion = nn.CrossEntropyLoss()
    scaler = torch.cuda.amp.GradScaler()

    losses, f1_scores = [], []

    for epoch in range(epochs):
        loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
        preds, labels, _ = evaluate_model(model, val_loader)
        acc = accuracy_score(labels, preds)
        f1 = f1_score(labels, preds, average='macro')
        print(f"Epoch {epoch+1}/{epochs} - Loss: {loss:.4f}, Val Acc: {acc:.4f}, Val F1: {f1:.4f}")
        losses.append(loss)
        f1_scores.append(f1)

    # Final evaluation on test set
    preds, labels, logits = evaluate_model(model, test_loader)

    # Metrics
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    try:
        roc_auc = roc_auc_score(labels, logits, multi_class='ovr')
    except Exception:
        roc_auc = None

    print(f"\nTest metrics for {model_name}:")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")
    if roc_auc:
        print(f"ROC AUC (OVR): {roc_auc:.4f}")

    # Confusion matrix and report
    cm = confusion_matrix(labels, preds)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, None]
    print("\nClassification Report:")
    print(classification_report(labels, preds, target_names=class_names))

    # Plot Loss & F1 curves
    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    plt.plot(range(1, epochs+1), losses, label='Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(f'{model_name} Training Loss')
    plt.legend()

    plt.subplot(1,2,2)
    plt.plot(range(1, epochs+1), f1_scores, label='F1 Score')
    plt.xlabel('Epoch')
    plt.ylabel('F1 Score')
    plt.title(f'{model_name} Validation F1 Score')
    plt.legend()
    plt.show()

    # Confusion matrix plot
    plt.figure(figsize=(7,6))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title(f'{model_name} Normalized Confusion Matrix')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.show()

    return model, acc, f1, roc_auc


In [17]:
# Initialize results dictionary
results = {}

# ViT-b32 baseline
model_vit_b32, acc_b32, f1_b32, roc_b32 = run_training_and_evaluation("ViT-b32", get_vit_b32)
results["ViT-b32"] = {"Model": model_vit_b32, "Accuracy": acc_b32, "F1": f1_b32, "ROC AUC": roc_b32}

# RViT
#model_rvit, acc_rvit, f1_rvit, roc_rvit = run_training_and_evaluation("RViT", get_rvit)
#results["RViT"] = {"Model": model_rvit, "Accuracy": acc_rvit, "F1": f1_rvit, "ROC AUC": roc_rvit}

# FTVT-I16
model_ftvt, acc_ftvt, f1_ftvt, roc_ftvt = run_training_and_evaluation("FTVT-I16", get_ftvt_i16)
results["FTVT-I16"] = {"Model": model_ftvt, "Accuracy": acc_ftvt, "F1": f1_ftvt, "ROC AUC": roc_ftvt}

# ViT-b32 + GViT regularization
#model_gvit, acc_gvit, f1_gvit, roc_gvit = run_training_and_evaluation("ViT-b32 + GViT", get_vit_b32, apply_reg=True)
#results["ViT-b32 + GViT"] = {"Model": model_gvit, "Accuracy": acc_gvit, "F1": f1_gvit, "ROC AUC": roc_gvit}

# ViT-I16 + SCA (Selective Cross-Attention)
#model_vit_sca, acc_sca, f1_sca, roc_sca = run_training_and_evaluation("ViT-I16 + SCA", get_vit_i16, apply_fs='sca')
#results["ViT-I16 + SCA"] = {"Model": model_vit_sca, "Accuracy": acc_sca, "F1": f1_sca, "ROC AUC": roc_sca}

# ViT-I16 + FCM (Feature Calibration Module)
# model_vit_fcm, acc_fcm, f1_fcm, roc_fcm = run_training_and_evaluation("ViT-I16 + FCM", get_vit_i16, apply_fs='fcm')
# results["ViT-I16 + FCM"] = {"Model": model_vit_fcm, "Accuracy": acc_fcm, "F1": f1_fcm, "ROC AUC": roc_fcm}

Training ViT-b32...


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch32-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\JARE WORKS\AppData\Local\Temp\ipykernel_17664\703267094.py:15: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
c:\Users\JARE WORKS\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\amp\grad_scaler.py:136: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(
C:\Users\JARE WORKS\AppData\Local\Temp\ipykernel_17664\408454897.py:7: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
c:\Users\JARE 

KeyboardInterrupt: 

In [ ]:
# FTVT-I16
model_ftvt, acc_ftvt, f1_ftvt, roc_ftvt = run_training_and_evaluation("FTVT-I16", get_ftvt_i16)
results["FTVT-I16"] = {"Model": model_ftvt, "Accuracy": acc_ftvt, "F1": f1_ftvt, "ROC AUC": roc_ftvt}

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, f1_score

# Initialize containers for evaluation results
all_metrics = []
all_class_reports = {}

# Assume `model_outputs` is a dict where keys are model names and values are (preds, labels, logits)
model_outputs = {
    "ViT-B32": (vit_preds, vit_labels, vit_logits),
    "RViT": (rvit_preds, rvit_labels, rvit_logits),
    "FTViT-I16": (ftvit_preds, ftvit_labels, ftvit_logits)
}

class_names = ['glioma', 'meningioma', 'notumor', 'pituitary']

# Save evaluation metrics, plots, and confusion matrices for all models
for model_name, (preds, labels, logits) in model_outputs.items():
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    roc = roc_auc_score(labels, logits, multi_class='ovr')

    # Store metrics
    all_metrics.append({
        "Model": model_name,
        "Accuracy": acc,
        "F1 Score": f1,
        "ROC AUC": roc
    })

    # Classification report
    report_dict = classification_report(labels, preds, output_dict=True, target_names=class_names)
    all_class_reports[model_name] = pd.DataFrame(report_dict).transpose()

    # Confusion matrix
    cm = confusion_matrix(labels, preds)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Normalized Confusion Matrix: {model_name}")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()
    plt.savefig(f"results/conf_matrix_{model_name}.png")
    plt.close()

# Convert metrics to DataFrame
metrics_df = pd.DataFrame(all_metrics)

# Save to Excel
excel_path = 'results/combined_metrics_report.xlsx'
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    metrics_df.to_excel(writer, index=False, sheet_name="Scores")
    for model_name, df in all_class_reports.items():
        df.to_excel(writer, sheet_name=f"{model_name} Report")

print(f"✅ All metrics and classification reports exported to {excel_path}")


In [ ]:
# Plot Accuracy, F1 Score, and ROC AUC across all models
metrics_df.set_index("Model")[["Accuracy", "F1 Score", "ROC AUC"]].plot(kind='bar', figsize=(10, 6))
plt.title("Evaluation Metrics Comparison")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("results/model_comparison_metrics.png")
plt.show()
